# WarpTracer — single-car benchmark
Choose a GPU runtime for CUDA timings. One car, four physics substeps per transition, 1080 LiDAR rays at 60 Hz. CPU runs also work. Setup reuses the same repository and uv environment as the driving notebook.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path("/content/WarpTracer-rigidbody")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch",
                    "performance-prototype", "https://github.com/yuywe/WarpTracer.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["git", "fetch", "origin", "performance-prototype:refs/remotes/origin/performance-prototype"], check=True)
if subprocess.run(["git", "show-ref", "--verify", "--quiet", "refs/heads/performance-prototype"]).returncode:
    subprocess.run(["git", "branch", "--no-track", "performance-prototype", "origin/performance-prototype"], check=True)
subprocess.run(["git", "switch", "performance-prototype"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "performance-prototype"], check=True)
os.environ["UV_PYTHON"] = sys.executable
subprocess.run(["uv", "sync", "--locked", "--extra", "viz"], check=True)


In [ ]:
physics = "lean"  # rerun with "newton" for the baseline
subprocess.run(["uv", "run", "--locked", "python", "-m", "warptracer.benchmark",
                "--physics", physics, "--backend", "both",
                "--output", f"outputs/{physics}-benchmark.json"], check=True)


In [ ]:
import json
report = json.loads(Path(f"outputs/{physics}-benchmark.json").read_text())
print(report["device_name"], report["physics_engine"])
for result in report["results"]:
    s = result["summary"]
    print(result["case"], result["backend"],
          round(s["median_physics_substeps_per_second"]), "physics substeps/s;",
          round(s["median_environment_transitions_per_second"]), "environment transitions/s")
print("Full report:", Path(f"outputs/{physics}-benchmark.json").resolve())
